In [24]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Torch: {torch.__version__}')
print(f'Device: {device}')

Torch: 2.11.0+cu128
Device: cuda


## Embedding, positional encoding, normalization, residual, masks

In [25]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :].requires_grad_(False)
        return self.dropout(x)


class LayerNormalization(nn.Module):
    def __init__(self, d_model: int, epsilon: float = 1e-6):
        super().__init__()
        self.epsilon = epsilon
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.alpha * (x - mean) / torch.sqrt(variance + self.epsilon) + self.bias


class RMSNorm(nn.Module):
    def __init__(self, d_model: int, epsilon: float = 1e-6):
        super().__init__()
        self.epsilon = epsilon
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.epsilon)
        return x * rms * self.weight


class ResidualConnection(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = RMSNorm(d_model)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))


def causal_mask(size: int) -> torch.Tensor:
    mask = torch.triu(torch.ones(1, size, size, dtype=torch.bool), diagonal=1)
    return ~mask


def create_src_mask(src: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)


def create_tgt_mask(tgt: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    tgt_pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)
    tgt_causal_mask = causal_mask(tgt.size(1)).type_as(tgt_pad_mask).to(tgt.device)
    return tgt_pad_mask & tgt_causal_mask


def create_masks(src: torch.Tensor, tgt: torch.Tensor, pad_idx: int = 0):
    return create_src_mask(src, pad_idx), create_tgt_mask(tgt, pad_idx)


def subsequent_mask(size: int) -> torch.Tensor:
    return causal_mask(size)

## Attention, SwiGLU feed forward, encoder

In [26]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch, num_kv_head, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states

    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_head, n_rep, seq_len, head_dim)
    return hidden_states.reshape(batch, num_kv_head * n_rep, seq_len, head_dim)


class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate = F.silu(self.w1(x))
        up = self.w2(x)
        return self.dropout(self.w3(gate * up))


class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, num_head: int, dropout: float, num_kv_head: int | None = None):
        super().__init__()
        num_kv_head = num_head if num_kv_head is None else num_kv_head

        if num_head <= 0:
            raise ValueError('num_head must be greater than 0')
        if num_kv_head <= 0:
            raise ValueError('num_kv_head must be greater than 0')
        if d_model % num_head != 0:
            raise ValueError('d_model must be divisible by num_head')
        if num_head % num_kv_head != 0:
            raise ValueError('num_head must be divisible by num_kv_head')

        self.d_model = d_model
        self.num_head = num_head
        self.num_kv_head = num_kv_head
        self.n_rep = num_head // num_kv_head
        self.d_k = d_model // num_head

        self.w_q = nn.Linear(d_model, num_head * self.d_k, bias=False)
        self.w_k = nn.Linear(d_model, num_kv_head * self.d_k, bias=False)
        self.w_v = nn.Linear(d_model, num_kv_head * self.d_k, bias=False)
        self.w_o = nn.Linear(num_head * self.d_k, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.attention_score = None

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout | None):
        d_k = query.shape[-1]
        attention_score = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            attention_score = attention_score.masked_fill(mask == 0, -1e9)

        attention_score = attention_score.softmax(dim=-1)
        if dropout is not None:
            attention_score = dropout(attention_score)

        return attention_score @ value, attention_score

    def forward(self, q, k, v, mask):
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        query = query.view(query.shape[0], query.shape[1], self.num_head, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.num_kv_head, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.num_kv_head, self.d_k).transpose(1, 2)

        key = repeat_kv(key, self.n_rep)
        value = repeat_kv(value, self.n_rep)

        x, self.attention_score = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.num_head * self.d_k)
        return self.w_o(x)


class EncoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionBlock, feed_forward_block: FeedForwardBlock, d_model: int, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connection = nn.ModuleList([ResidualConnection(d_model, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connection[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connection[1](x, self.feed_forward_block)
        return x


class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList, d_model: int):
        super().__init__()
        self.layers = layers
        self.norm = RMSNorm(d_model)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

## Decoder, projection, transformer, builder

In [27]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        self_attention: MultiHeadAttentionBlock,
        cross_attention: MultiHeadAttentionBlock,
        feed_forward_block: FeedForwardBlock,
        d_model: int,
        dropout: float,
    ):
        super().__init__()
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.feed_forward_block = feed_forward_block
        self.residual_connection = nn.ModuleList([ResidualConnection(d_model, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connection[0](x, lambda x: self.self_attention(x, x, x, tgt_mask))
        x = self.residual_connection[1](x, lambda x: self.cross_attention(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connection[2](x, self.feed_forward_block)
        return x


class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList, d_model: int):
        super().__init__()
        self.layers = layers
        self.norm = RMSNorm(d_model)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)


class ProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.projection = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        return torch.log_softmax(self.projection(x), dim=-1)


class Transformer(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        src_embed: InputEmbedding,
        tgt_embed: InputEmbedding,
        src_pos: PositionalEncoding,
        tgt_pos: PositionalEncoding,
        projection_layer: ProjectionLayer,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)

    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        tgt = self.tgt_embed(tgt)
        tgt = self.tgt_pos(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        return self.projection_layer(x)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(encoder_output, src_mask, tgt, tgt_mask)
        return self.project(decoder_output)


def build_transformer(
    src_vocab_size: int,
    tgt_vocab_size: int,
    src_seq_len: int,
    tgt_seq_len: int,
    d_model: int = 512,
    N: int = 6,
    h: int = 8,
    dropout: float = 0.1,
    d_ff: int = 2048,
    num_kv_head: int | None = None,
) -> Transformer:
    num_kv_head = h if num_kv_head is None else num_kv_head

    src_embed = InputEmbedding(d_model, src_vocab_size)
    tgt_embed = InputEmbedding(d_model, tgt_vocab_size)

    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    encoder_blocks = []
    for _ in range(N):
        self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout, num_kv_head=num_kv_head)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        encoder_blocks.append(EncoderBlock(self_attention_block, feed_forward_block, d_model, dropout))

    decoder_blocks = []
    for _ in range(N):
        self_attention_block = MultiHeadAttentionBlock(d_model, h, dropout, num_kv_head=num_kv_head)
        cross_attention_block = MultiHeadAttentionBlock(d_model, h, dropout, num_kv_head=num_kv_head)
        feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
        decoder_blocks.append(
            DecoderBlock(self_attention_block, cross_attention_block, feed_forward_block, d_model, dropout)
        )

    encoder = Encoder(nn.ModuleList(encoder_blocks), d_model)
    decoder = Decoder(nn.ModuleList(decoder_blocks), d_model)
    projection_layer = ProjectionLayer(d_model, tgt_vocab_size)
    transformer = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer)

    for parameter in transformer.parameters():
        if parameter.dim() > 1:
            nn.init.xavier_uniform_(parameter)

    return transformer

## Quick run

In [28]:
PAD_IDX = 0
SRC_VOCAB_SIZE = 100
TGT_VOCAB_SIZE = 120
SRC_SEQ_LEN = 12
TGT_SEQ_LEN = 10
BATCH_SIZE = 4

model = build_transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    src_seq_len=SRC_SEQ_LEN,
    tgt_seq_len=TGT_SEQ_LEN,
    d_model=64,
    N=2,
    h=4,
    num_kv_head=2,
    dropout=0.1,
    d_ff=128,
).to(device)

src = torch.randint(1, SRC_VOCAB_SIZE, (BATCH_SIZE, SRC_SEQ_LEN), device=device)
tgt_input = torch.randint(1, TGT_VOCAB_SIZE, (BATCH_SIZE, TGT_SEQ_LEN), device=device)

src[:, -2:] = PAD_IDX
tgt_input[:, -1:] = PAD_IDX

src_mask, tgt_mask = create_masks(src, tgt_input, pad_idx=PAD_IDX)
log_probs = model(src, tgt_input, src_mask, tgt_mask)

print('src:', tuple(src.shape))
print('tgt_input:', tuple(tgt_input.shape))
print('src_mask:', tuple(src_mask.shape))
print('tgt_mask:', tuple(tgt_mask.shape))
print('log_probs:', tuple(log_probs.shape))

assert log_probs.shape == (BATCH_SIZE, TGT_SEQ_LEN, TGT_VOCAB_SIZE)

src: (4, 12)
tgt_input: (4, 10)
src_mask: (4, 1, 1, 12)
tgt_mask: (4, 1, 10, 10)
log_probs: (4, 10, 120)


## One dummy training step

In [29]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.NLLLoss(ignore_index=PAD_IDX)

target = torch.randint(1, TGT_VOCAB_SIZE, (BATCH_SIZE, TGT_SEQ_LEN), device=device)
target[:, -1:] = PAD_IDX

optimizer.zero_grad(set_to_none=True)
log_probs = model(src, tgt_input, src_mask, tgt_mask)
loss = criterion(log_probs.reshape(-1, TGT_VOCAB_SIZE), target.reshape(-1))
loss.backward()
optimizer.step()

num_params = sum(p.numel() for p in model.parameters())
print(f'loss: {loss.item():.4f}')
print(f'parameters: {num_params:,}')

loss: 5.0254
parameters: 194,560



## Train thử với dataset `article → summary`

Phần này gắn dataset trong `eda.ipynb` vào Transformer scratch hiện tại.  
Mục tiêu không phải tạo model tóm tắt thật tốt ngay, mà là chứng minh pipeline chạy đúng:

1. Đọc `train.parquet`.
2. Tokenize văn bản.
3. Build vocabulary.
4. Tạo `Dataset` / `DataLoader` cho bài toán summarization.
5. Train vài batch và kiểm tra loss.
6. Sinh thử summary bằng greedy decoding.

> Đặt file `train.parquet` ở một trong các vị trí: `train.parquet`, `data/train.parquet`, hoặc `../data/train.parquet`.


In [30]:
# Nếu môi trường của bạn chưa đọc được parquet, bỏ comment dòng dưới rồi chạy một lần.
%pip install pandas pyarrow

In [31]:

from pathlib import Path
import re
from collections import Counter

import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Token đặc biệt dùng chung cho source và target vocabulary
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

# Các tham số nhỏ để train thử nhanh trên CPU/GPU yếu.
# Có thể tăng dần khi đã chắc pipeline chạy đúng.
MAX_VOCAB_SIZE = 12_000
MIN_FREQ = 2
SRC_SEQ_LEN = 160
TGT_SEQ_LEN = 64
BATCH_SIZE = 4

# Chỉ lấy một phần dataset để demo loss giảm nhanh hơn.
# Đổi thành None nếu muốn dùng toàn bộ train split.
QUICK_TRAIN_SAMPLES = 512
QUICK_VALID_SAMPLES = 128

print("Device:", device)


Device: cuda


In [32]:
!pip install -q fastparquet

In [33]:
from pathlib import Path
import pandas as pd

def find_data_path(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("data") / filename,
        Path("..") / "data" / filename,
        Path("/mnt/data") / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy {filename}. Hãy đặt file ở một trong các vị trí: "
        f"{[str(c) for c in candidates]}"
    )

def inspect_parquet_file(path: Path):
    print("Dataset path:", path.resolve())
    print("File size:", round(path.stat().st_size / 1024 / 1024, 2), "MB")

    with open(path, "rb") as f:
        head = f.read(300)
        f.seek(max(0, path.stat().st_size - 20))
        tail = f.read()

    print("First bytes:", head[:120])
    print("Last bytes:", tail)

    if path.stat().st_size < 1024:
        raise ValueError(
            "File quá nhỏ, có thể đây không phải dataset thật hoặc file bị tải thiếu."
        )

    if head.startswith(b"version https://git-lfs.github.com/spec/v1"):
        raise ValueError(
            "File này là Git LFS pointer, không phải dataset thật. "
            "Bạn cần tải file bằng Git LFS hoặc tải dataset gốc."
        )

    if b"<html" in head.lower() or b"<!doctype html" in head.lower():
        raise ValueError(
            "File này có vẻ là HTML, không phải Parquet. "
            "Có thể bạn đã tải nhầm trang web thay vì file dataset."
        )

    if head.startswith(b"PK"):
        raise ValueError(
            "File này có vẻ là file ZIP. Hãy giải nén trước, rồi đọc file parquet bên trong."
        )

    if not head.startswith(b"PAR1") or not tail.endswith(b"PAR1"):
        raise ValueError(
            "File không có định dạng Parquet hợp lệ. "
            "Parquet file thường bắt đầu và kết thúc bằng bytes PAR1."
        )

train_path = find_data_path("train.parquet")
inspect_parquet_file(train_path)

try:
    df = pd.read_parquet(train_path, engine="pyarrow")
except Exception as e:
    print("pyarrow đọc lỗi:", repr(e))
    print("Thử đọc bằng fastparquet...")
    df = pd.read_parquet(train_path, engine="fastparquet")

required_cols = {"article", "summary"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(
        f"Dataset thiếu cột bắt buộc: {missing_cols}. "
        "Cần có article và summary."
    )

df = df[["article", "summary"]].dropna()
df = df[df["article"].astype(str).str.strip().ne("")]
df = df[df["summary"].astype(str).str.strip().ne("")]
df = df.drop_duplicates().reset_index(drop=True)

print(f"Rows after cleaning: {len(df):,}")
display(df.head(3))

Dataset path: /content/data/train.parquet
File size: 17.8 MB
First bytes: b'PAR1\x15\x04\x15\xd4\xf7\xc8\x02\x15\xac\xcb\xab\x01L\x15\xd0\x0f\x15\x00\x12\x00\x00\xea\xbb\xa4\x01\xf0\xaa3\x08\x00\x00G\xe1\xba\xa7n 20 s\xe1\xbb\xb1 ki\xe1\xbb\x87n \xc4\x91\xc6\xb0\xe1\xbb\xa3c t\xe1\xbb\x95 ch\xe1\xbb\xa9c tr\xc3\xaan to\xc3\xa0n th\xc3\xa0nh ph\xe1\xbb\x91, k\xc3\xa9o d\xc3\xa0i t\xe1\xbb\xab 19'
Last bytes: b'1.0\x19,\x1c\x00\x00\x1c\x00\x00\x00\xc2\x1b\x01\x00PAR1'
Rows after cleaning: 10,775


,article,summary
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...



### Tokenizer đơn giản

Tokenizer dưới đây dùng regex nên đủ để kiểm tra pipeline. Với tiếng Việt thật sự, sau khi demo chạy ổn bạn có thể thay bằng SentencePiece/BPE để chất lượng tốt hơn.


In [34]:

TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text: str) -> list[str]:
    text = str(text).lower().strip()
    return TOKEN_PATTERN.findall(text)

print(tokenize("Hôm nay, mô hình Transformer chạy thử với dữ liệu tiếng Việt."))


['hôm', 'nay', ',', 'mô', 'hình', 'transformer', 'chạy', 'thử', 'với', 'dữ', 'liệu', 'tiếng', 'việt', '.']


In [35]:

def build_vocab(dataframe: pd.DataFrame, max_vocab_size: int, min_freq: int = 1):
    counter = Counter()

    for col in ["article", "summary"]:
        for text in dataframe[col].astype(str):
            counter.update(tokenize(text))

    # Bỏ token quá hiếm để giảm nhiễu và giảm kích thước model.
    tokens = [token for token, freq in counter.most_common() if freq >= min_freq]
    tokens = tokens[: max_vocab_size - len(SPECIAL_TOKENS)]

    itos = SPECIAL_TOKENS + tokens
    stoi = {token: idx for idx, token in enumerate(itos)}
    return stoi, itos, counter

stoi, itos, token_counter = build_vocab(df, MAX_VOCAB_SIZE, MIN_FREQ)
VOCAB_SIZE = len(itos)

print(f"Vocab size: {VOCAB_SIZE:,}")
print("Most common tokens:", token_counter.most_common(20))


Vocab size: 12,000
Most common tokens: [(',', 346129), ('.', 266893), ('và', 91055), ('của', 57596), ('có', 55936), ('trong', 51740), ('với', 47106), ('các', 46833), ('là', 46274), ('cho', 41875), ('được', 41480), ('một', 39954), ('"', 39273), ('không', 35605), ('người', 34672), ('năm', 33309), ('ở', 33137), ('-', 32880), ('khi', 30957), ('công', 30952)]


In [36]:

def tokens_to_ids(tokens: list[str]) -> list[int]:
    return [stoi.get(token, UNK_IDX) for token in tokens]


def encode_source(text: str, max_len: int = SRC_SEQ_LEN) -> list[int]:
    ids = tokens_to_ids(tokenize(text))[:max_len]
    ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids


def encode_target(summary: str, max_len: int = TGT_SEQ_LEN) -> tuple[list[int], list[int]]:
    """Tạo cặp target cho decoder.

    tgt_input: <bos> y1 y2 y3 ...
    tgt_label: y1 y2 y3 ... <eos>
    """
    summary_ids = tokens_to_ids(tokenize(summary))[: max_len - 1]

    tgt_input = [BOS_IDX] + summary_ids
    tgt_label = summary_ids + [EOS_IDX]

    tgt_input = tgt_input + [PAD_IDX] * (max_len - len(tgt_input))
    tgt_label = tgt_label + [PAD_IDX] * (max_len - len(tgt_label))

    return tgt_input, tgt_label


def decode_ids(ids: list[int]) -> str:
    tokens = []
    for idx in ids:
        if idx == EOS_IDX:
            break
        if idx in {PAD_IDX, BOS_IDX}:
            continue
        tokens.append(itos[idx] if 0 <= idx < len(itos) else UNK_TOKEN)

    text = " ".join(tokens)
    # Detokenize thô cho dễ đọc hơn một chút.
    text = re.sub(r"\s+([,.;:!?%)\]\}])", r"\1", text)
    text = re.sub(r"([([{])\s+", r"\1", text)
    return text

sample_tgt_input, sample_tgt_label = encode_target(df.loc[0, "summary"])
print("tgt_input[:15]:", sample_tgt_input[:15])
print("tgt_label[:15]:", sample_tgt_label[:15])
print("decoded label:", decode_ids(sample_tgt_label[:30]))


tgt_input[:15]: [2, 270, 193, 256, 186, 150, 428, 56, 296, 24, 745, 37, 169, 36, 156]
tgt_label[:15]: [270, 193, 256, 186, 150, 428, 56, 296, 24, 745, 37, 169, 36, 156, 37]
decoded label: hà nội tổ chức gần 20 sự kiện từ 19 / 4 đến 10 / 5, bao gồm lễ hội du lịch hà nội 2024, các triển lãm


In [37]:

class ArticleSummaryDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]
        src = encode_source(row["article"], SRC_SEQ_LEN)
        tgt_input, tgt_label = encode_target(row["summary"], TGT_SEQ_LEN)

        return {
            "src": torch.tensor(src, dtype=torch.long),
            "tgt_input": torch.tensor(tgt_input, dtype=torch.long),
            "tgt_label": torch.tensor(tgt_label, dtype=torch.long),
        }


# Shuffle thủ công để không cần sklearn.
df_shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
valid_size = max(1, int(len(df_shuffled) * 0.1))
valid_df = df_shuffled.iloc[:valid_size].reset_index(drop=True)
train_df = df_shuffled.iloc[valid_size:].reset_index(drop=True)

if QUICK_TRAIN_SAMPLES is not None:
    train_df = train_df.head(QUICK_TRAIN_SAMPLES)
if QUICK_VALID_SAMPLES is not None:
    valid_df = valid_df.head(QUICK_VALID_SAMPLES)

train_dataset = ArticleSummaryDataset(train_df)
valid_dataset = ArticleSummaryDataset(valid_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print("Train rows:", len(train_dataset))
print("Valid rows:", len(valid_dataset))
print("src shape:", tuple(batch["src"].shape))
print("tgt_input shape:", tuple(batch["tgt_input"].shape))
print("tgt_label shape:", tuple(batch["tgt_label"].shape))


Train rows: 512
Valid rows: 128
src shape: (4, 160)
tgt_input shape: (4, 64)
tgt_label shape: (4, 64)



### Build model nhỏ để train thử

Ở đây dùng model nhỏ hơn Transformer gốc để chạy được trên laptop.  
Sau khi pipeline đúng, bạn có thể tăng `d_model`, `N`, `d_ff`, `SRC_SEQ_LEN`, `TGT_SEQ_LEN`.


In [38]:

model = build_transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    src_seq_len=SRC_SEQ_LEN,
    tgt_seq_len=TGT_SEQ_LEN,
    d_model=128,
    N=2,
    h=4,
    dropout=0.1,
    d_ff=512,
    num_kv_head=None,  # None nghĩa là Multi-Head Attention chuẩn, không dùng GQA.
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.NLLLoss(ignore_index=PAD_IDX)

num_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_params:,}")


Parameters: 5,789,184


In [39]:

def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device) for key, value in batch.items()}


def compute_loss(model: nn.Module, batch: dict) -> torch.Tensor:
    src = batch["src"]
    tgt_input = batch["tgt_input"]
    tgt_label = batch["tgt_label"]

    src_mask, tgt_mask = create_masks(src, tgt_input, pad_idx=PAD_IDX)
    log_probs = model(src, tgt_input, src_mask, tgt_mask)

    loss = criterion(
        log_probs.reshape(-1, log_probs.size(-1)),
        tgt_label.reshape(-1),
    )
    return loss


def train_one_epoch(model: nn.Module, dataloader: DataLoader, max_batches: int | None = None) -> float:
    model.train()
    total_loss = 0.0
    total_batches = 0

    for batch_idx, batch in enumerate(dataloader, start=1):
        if max_batches is not None and batch_idx > max_batches:
            break

        batch = move_batch_to_device(batch, device)
        loss = compute_loss(model, batch)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

        if batch_idx == 1 or batch_idx % 20 == 0:
            print(f"batch {batch_idx:03d} | train loss: {loss.item():.4f}")

    return total_loss / max(1, total_batches)


@torch.no_grad()
def evaluate(model: nn.Module, dataloader: DataLoader, max_batches: int | None = 20) -> float:
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch_idx, batch in enumerate(dataloader, start=1):
        if max_batches is not None and batch_idx > max_batches:
            break

        batch = move_batch_to_device(batch, device)
        loss = compute_loss(model, batch)

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)


In [40]:

# Train demo: mục tiêu là thấy loss chạy được và thường giảm qua các batch/epoch.
# Nếu dùng CPU, giữ NUM_EPOCHS nhỏ. Nếu có GPU, có thể tăng lên 3-5.
NUM_EPOCHS = 2
MAX_TRAIN_BATCHES = 100
MAX_VALID_BATCHES = 20

history = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    train_loss = train_one_epoch(model, train_loader, max_batches=MAX_TRAIN_BATCHES)
    valid_loss = evaluate(model, valid_loader, max_batches=MAX_VALID_BATCHES)

    history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})
    print(f"Epoch {epoch} done | train loss: {train_loss:.4f} | valid loss: {valid_loss:.4f}")

pd.DataFrame(history)



Epoch 1/2
batch 001 | train loss: 9.4127
batch 020 | train loss: 8.7234
batch 040 | train loss: 7.9877
batch 060 | train loss: 7.4997
batch 080 | train loss: 7.2818
batch 100 | train loss: 6.9695
Epoch 1 done | train loss: 7.8942 | valid loss: 7.0021

Epoch 2/2
batch 001 | train loss: 6.8812
batch 020 | train loss: 6.7583
batch 040 | train loss: 6.9389
batch 060 | train loss: 6.5348
batch 080 | train loss: 6.6298
batch 100 | train loss: 6.6353
Epoch 2 done | train loss: 6.8173 | valid loss: 6.8907


,epoch,train_loss,valid_loss
0,1,7.89415,7.002124
1,2,6.81727,6.890741



### Sinh thử summary bằng greedy decoding

Vì model chỉ train thử rất ít batch nên output có thể còn lặp hoặc chưa hay. Mục tiêu của cell này là kiểm tra luồng inference encoder-decoder.


In [41]:

@torch.no_grad()
def greedy_decode(model: nn.Module, article: str, max_len: int = TGT_SEQ_LEN) -> str:
    model.eval()

    src = torch.tensor([encode_source(article, SRC_SEQ_LEN)], dtype=torch.long, device=device)
    src_mask = create_src_mask(src, pad_idx=PAD_IDX)
    encoder_output = model.encode(src, src_mask)

    ys = torch.tensor([[BOS_IDX]], dtype=torch.long, device=device)

    for _ in range(max_len - 1):
        tgt_mask = create_tgt_mask(ys, pad_idx=PAD_IDX)
        decoder_output = model.decode(encoder_output, src_mask, ys, tgt_mask)
        log_probs = model.project(decoder_output[:, -1:, :])
        next_token = torch.argmax(log_probs[:, -1, :], dim=-1).item()

        ys = torch.cat(
            [ys, torch.tensor([[next_token]], dtype=torch.long, device=device)],
            dim=1,
        )

        if next_token == EOS_IDX:
            break

    return decode_ids(ys.squeeze(0).tolist())

sample = valid_df.iloc[0]
print("ARTICLE:")
print(str(sample["article"])[:1000], "...")
print("\nGOLD SUMMARY:")
print(sample["summary"])
print("\nMODEL SUMMARY:")
print(greedy_decode(model, sample["article"]))


ARTICLE:
Trong 11 tháng qua, Trung tâm Dịch vụ việc làm Hà Nội giải quyết cho hơn 80.200 người nhận trợ cấp thất nghiệp, 730 người được hỗ trợ học nghề. Ước tính cả năm, hồ sơ hưởng bảo hiểm thất nghiệp tăng khoảng 20% so với 2022. Hồ sơ gửi về trung tâm hồi giữa năm dao động 9.000-10.000 do làn sóng mất việc, giãn việc, giảm giờ làm tăng mạnh. Từ tháng 10 đến nay, lượng hồ sơ gửi về giảm còn khoảng 6.000 mỗi tháng. "Đây là tín hiệu đáng mừng, cho thấy tình trạng lao động mất việc những tháng cuối năm đã giảm nhiệt, không gay gắt như thời gian trước", bà Vũ Thị Thanh Liễu, Phó giám đốc Trung tâm, đánh giá. Độ tuổi hưởng trợ cấp thất nghiệp cao nhất là nhóm nữ từ 25 đến dưới 40, chiếm gần 40%, nam cùng độ tuổi chiếm 29%; nữ trên 40 tuổi chiếm 12,5%, nam cùng độ tuổi khoảng 10%. Điều này cho thấy nguy cơ mất việc ở lao động trung tuổi ngày càng tăng. Trên 35 tuổi, nhiều người bắt đầu suy giảm khả năng làm việc và nhiều doanh nghiệp không muốn tuyển dụng. 50% người nộp hồ sơ là lao động c


## Gợi ý khi báo cáo

Bạn có thể viết:

> Nhóm đã gắn dataset `article → summary` vào mô hình Transformer encoder-decoder tự cài đặt. Dữ liệu được tokenize, ánh xạ thành token IDs, đưa qua `Dataset` và `DataLoader`. Trong quá trình huấn luyện thử, mô hình nhận `article` ở encoder và nhận chuỗi `summary` đã dịch phải ở decoder. Loss được tính giữa output decoder và nhãn summary thật, bỏ qua token padding. Kết quả train thử cho thấy pipeline hoạt động đúng, loss có thể tính toán và lan truyền ngược qua toàn bộ mô hình.

Lưu ý: tokenizer regex chỉ dùng để demo. Nếu muốn tăng chất lượng tóm tắt tiếng Việt, nên thay bằng SentencePiece/BPE hoặc fine-tune mô hình pretrained như ViT5/BARTPho.
